# Fase 12 — Auditoría de robustez y estabilidad

Esta fase evalúa si las recomendaciones validadas de la fase 11 permanecen estables bajo filtros de evidencia, umbrales alternativos y eliminación secuencial de fuentes.

Principios metodológicos:

- los hallazgos no se tratan como estudios independientes;
- la fuente es la unidad principal de independencia;
- implementación no equivale a efectividad;
- estabilidad analítica no equivale a eficacia de manejo;
- una recomendación sensible se conserva como resultado, no se elimina automáticamente;
- todos los productos mantienen trazabilidad hacia `option_id`, `recommendation_id` y `source_id`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from evidence_review.robustness_audit import (
    build_evidence_independence_summary,
    build_recommendation_stability,
    build_robustness_summary,
    build_sensitivity_scenario_results,
    build_source_concentration_summary,
    build_source_influence_matrix,
    leave_one_source_out_summary,
    load_robustness_config,
    read_csv_robust,
    scenario_retention_summary,
    split_stability_outputs,
    validate_robustness_outputs,
)

CONFIG_PATH = ROOT / 'config' / 'robustness_audit.yml'
config = load_robustness_config(CONFIG_PATH, project_root=ROOT)
decision_config = config['decision_support_config']
paths = config['paths']

print(f'Project root: {ROOT}')
print(f'Configuration: {CONFIG_PATH.relative_to(ROOT)}')
print(f'Configured scenarios: {len(config["scenarios"])}')


## 1. Cargar productos validados de las fases 10 y 11

La auditoría utiliza la matriz de 97 hallazgos, el portafolio de opciones y únicamente recomendaciones con revisión experta `accepted` o `corrected`.


In [ ]:
matrix, matrix_encoding = read_csv_robust(ROOT / paths['synthesis_matrix_csv'])
portfolio, portfolio_encoding = read_csv_robust(ROOT / paths['management_portfolio_csv'])
recommendations, recommendations_encoding = read_csv_robust(
    ROOT / paths['recommendation_validated_csv']
)

if matrix.empty:
    raise ValueError('The validated evidence synthesis matrix is empty.')
if portfolio.empty:
    raise ValueError('The management option portfolio is empty.')
if recommendations.empty:
    raise ValueError('No expert-validated recommendations are available.')

invalid_review = recommendations.loc[
    ~recommendations['expert_review_status'].isin(['accepted', 'corrected'])
]
if not invalid_review.empty:
    raise ValueError('Validated recommendation input contains unreviewed rows.')

print(f'Evidence findings: {len(matrix)} [{matrix_encoding}]')
print(f'Independent sources: {matrix["source_id"].nunique()}')
print(f'Management options: {len(portfolio)} [{portfolio_encoding}]')
print(f'Validated recommendations: {len(recommendations)} [{recommendations_encoding}]')


## 2. Escenarios de sensibilidad

Se reconstruye el portafolio bajo ocho especificaciones. Los escenarios describen sensibilidad a supuestos; no son probabilidades ni réplicas independientes.


In [ ]:
scenario_results = build_sensitivity_scenario_results(
    matrix,
    decision_config,
    config,
)
scenario_summary = scenario_retention_summary(scenario_results, config)

print(f'Scenario-option rows: {len(scenario_results)}')
display(
    scenario_results.groupby('scenario_id', as_index=False).agg(
        options_present=('option_present', 'sum'),
        options_retained=('retained', 'sum'),
        evidence_rows=('evidence_rows_available', 'max'),
    )
)
display(scenario_summary.sort_values('scenario_retention_rate').head(20))


## 3. Influencia de fuentes e independencia

Se elimina una fuente a la vez y se reconstruye el portafolio. También se calcula concentración mediante participación de la fuente dominante, HHI y número efectivo de fuentes.


In [ ]:
source_influence = build_source_influence_matrix(
    matrix,
    decision_config,
    config,
)
loso_summary = leave_one_source_out_summary(source_influence)
source_concentration = build_source_concentration_summary(matrix, config)
evidence_independence = build_evidence_independence_summary(
    portfolio,
    source_concentration,
)

print(f'Leave-one-source-out rows: {len(source_influence)}')
print(f'Source-concentration rows: {len(source_concentration)}')
display(loso_summary.sort_values('leave_one_source_out_retention').head(20))
display(source_concentration.sort_values('top_source_share', ascending=False).head(20))


## 4. Estabilidad de recomendaciones

Cada recomendación validada recibe una clase de estabilidad. Esta clasificación no modifica su aceptación experta y no demuestra efectividad.


In [ ]:
stability = build_recommendation_stability(
    recommendations,
    portfolio,
    scenario_summary,
    loso_summary,
    source_concentration,
    config,
)
stability_groups = split_stability_outputs(stability)

display(
    stability['stability_class']
    .value_counts(dropna=False)
    .rename_axis('stability_class')
    .reset_index(name='recommendations')
)
display(
    stability[
        [
            'management_measure',
            'base_readiness_class',
            'scenario_retention_rate',
            'leave_one_source_out_retention',
            'top_source_share',
            'implementation_dependency',
            'stability_class',
        ]
    ].sort_values(['stability_class', 'management_measure'])
)


## 5. Validación estructural

La validación comprueba identidades únicas, cobertura de escenarios, tasas, clases controladas y correspondencia con las recomendaciones revisadas.


In [ ]:
issues = validate_robustness_outputs(
    scenario_results,
    source_influence,
    source_concentration,
    stability,
    config,
)
summary = build_robustness_summary(
    matrix,
    scenario_results,
    source_influence,
    source_concentration,
    stability,
    issues,
)

print(f'Validation issues: {len(issues)}')
display(issues.head(100))
display(summary)


## 6. Exportar productos


In [ ]:
outputs = {
    paths['sensitivity_results_csv']: scenario_results,
    paths['recommendation_stability_csv']: stability,
    paths['source_influence_csv']: source_influence,
    paths['source_concentration_csv']: source_concentration,
    paths['evidence_independence_csv']: evidence_independence,
    paths['robust_recommendations_csv']: stability_groups['robust'],
    paths['sensitive_recommendations_csv']: stability_groups['sensitive'],
    paths['issues_csv']: issues,
    paths['summary_csv']: summary,
}

for relative_path, frame in outputs.items():
    output_path = ROOT / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'{output_path.relative_to(ROOT)}: {len(frame)}')


## Criterio para avanzar a la fase 13

La generación de figuras, tablas y reporte final puede comenzar cuando:

1. `Validation issues = 0`;
2. los ocho escenarios están representados para cada opción base;
3. el análisis leave-one-source-out cubre todas las fuentes independientes;
4. todas las recomendaciones validadas tienen una clase de estabilidad;
5. los hallazgos, fuentes, implementación y efectividad permanecen diferenciados;
6. las recomendaciones sensibles se reportan explícitamente y no se ocultan;
7. los archivos de `data/interim/` se conservan fuera de Git.

La siguiente fase será `notebooks/13_build_scientific_products.ipynb`.
